# Stage 5: one event, one shared mission plan

This notebook reads recorded evidence only. Run All sends no requests, creates no jobs and spends no credits. The live application uses a separate durable session worker. These are synthetic paused-clock results, not a flight controller.

In [1]:
from pathlib import Path
import json
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/verification/unified-planner-stage5-2026-11-15.json").exists())
evidence = json.loads((root / "docs/verification/unified-planner-stage5-2026-11-15.json").read_text())
print("Recorded session:", evidence["session_id"])
print("Synthetic:", evidence["synthetic"])

Recorded session: cb1a1f49-b3c9-451b-8ed1-ec0641c71b2b
Synthetic: True


## 1. Input change and plan identity

The landing exclusion advances the simulation to second 8. Revision 0 becomes history. Revision 1 accepts a separately validated continuation. The Campaign, Self-healing and Solver Arena views use the same session and plan ID.

In [2]:
before, after = evidence["initial_plan"], evidence["event_plan"]
for label, plan in (("Before", before), ("After", after)):
    print(label, "revision", plan["revision"], "target", plan["target_id"], "source", plan["source"])
    print("Plan:", plan["plan_id"])
    print("Input:", plan["snapshot_sha256"])
print("Event:", evidence["event"])

Before revision 0 target primary source classical
Plan: d55c8c8c22d88425a27e0b247660a03229c1755daba9e26e6789bda984090572
Input: e821387bb2fa368f6270e060e3f9258d835e9bbb37960b8dbe0e88d233ded6ce
After revision 1 target remote source classical
Plan: a8ee238e44635ccad6eab4144e2af9f6b411ae1878790a6a5f34d3445cbfad9a
Input: 05b634edd32247fb1d0d41505fabcae48c4ad1baa737a4e24b4cd0c5263da1ac
Event: {'request_id': 'stage5-obstacle', 'expected_revision': 0, 'mode': 'local_experiment', 'ttl_s': 30.0, 'budget': {'depth': 1, 'evaluations': 32, 'shots': 64, 'seed': 17, 'max_qubits': 4, 'max_completions': 2, 'completion_limit_s': 5.0}, 'event_id': 'stage5-obstacle', 'elapsed_s': 8.0, 'start_downrange_m': -5.0, 'end_downrange_m': 5.0}


## 2. All missions and shared communications

The active lander is physically propagated. The dependent rover and other lander retain their native campaign constraints. Rover scheduling is not a simulated rover trajectory.

In [3]:
for mission in after["timeline"]["missions"]:
    print(mission["mission_id"], mission["status"], mission["landing_utc"], "delivered bits", mission["delivered_bits"])
for allocation in after["timeline"]["allocations"]:
    print(allocation["mission_id"], allocation["station_id"], allocation["start_utc"], allocation["end_utc"])
print("Remaining propellant (kg):", after["physical"]["fine"]["state"]["propellant_kg"])
print("Physical samples passed:", after["physical"]["uncertainty"]["pass_count"])

landing nominal 2030-01-01T00:00:34.999988Z delivered bits 60.0
other nominal 2030-01-01T00:01:00Z delivered bits 60.0
rover nominal 2030-01-01T00:01:45Z delivered bits 60.0
other dss-alpha 2030-01-01T00:01:00Z 2030-01-01T00:02:00Z
landing dss-alpha 2030-01-01T00:02:00Z 2030-01-01T00:03:00Z
rover dss-alpha 2030-01-01T00:03:00Z 2030-01-01T00:04:00Z
Remaining propellant (kg): 79.66869858090342
Physical samples passed: 8


## 3. Separate timing and matched proposal evidence

Queue and final independent validation are outside aggregate calculation time. Preparation, baseline, proposal and completion are substeps of calculation; do not add them twice. Raw local feasibility does not imply full-plan success or quantum advantage.

In [4]:
job = evidence["job"]
comparison = job["comparison"]
print({k: job[k] for k in ("status", "queue_s", "compute_s", "validation_s")})
print("Decision:", comparison["selection_reason"])
print("Preparation (s):", comparison["preparation_s"], "baseline (s):", comparison["baseline_s"])
for method in comparison["methods"]:
    print(method["method"], method["raw_counts"])
    for attempt in method["attempts"]:
        print(attempt["option_ids"], attempt["status"], "completion (s)", attempt["completion_s"], "admission (s)", attempt["admission_s"])

{'status': 'accepted', 'queue_s': 0.06077933311462402, 'compute_s': 2.767927251996298, 'validation_s': 0.7951404170016758}
Decision: no_strict_full_plan_improvement
Preparation (s): 0.3881113750030636 baseline (s): 0.3978784170030849
qaoa {'land-remote': 64}
['land-remote'] accepted completion (s) 0.01221991700003855 admission (s) 0.3765793759957887
random {'land-backup': 24, 'land-remote': 40}
['land-remote'] accepted completion (s) 0.012214499998663086 admission (s) 0.38656458300101804
['land-backup'] accepted completion (s) 0.012421334002283402 admission (s) 0.3682392920018174
greedy {'land-remote': 32, 'land-backup': 32}
['land-backup'] accepted completion (s) 0.013597999997728039 admission (s) 0.39677941700210795
['land-remote'] accepted completion (s) 0.012239958996360656 admission (s) 0.3735704999999143


## 4. Optional recorded QPU evidence

The hardware trial is an explicit offline experiment with a different wall clock. It is never auto-applied to an expired flight session. Read the report limitations; a one-qubit experiment cannot establish entanglement-based advantage.

In [5]:
hardware_path = root / "docs/verification/unified-stage5-hardware-2026-11-15.json"
if hardware_path.exists():
    hardware = json.loads(hardware_path.read_text())
    print(json.dumps(hardware, indent=2))
else:
    print("No completed hardware evidence recorded yet. No live request was made.")

No completed hardware evidence recorded yet. No live request was made.
